# vitok 01 — Data + tokenizers (CPU)

**Settings:** Accelerator = None (CPU), Internet = On. Attach the `vitok-code` dataset, or set `VITOK_REPO`.

Run with **Save Version → Save & Run All**. When it finishes, create a Kaggle Dataset named **`vitok-data`** from this notebook's output. Notebook 02 reads it.

Covers: data, the four tokenizers with the Gate 1 compression check, and the minimal pairs.

In [ ]:
VITOK_REPO = "" # e.g. "https://github.com/<you>/nanovitok.git"; leave empty when the vitok-code dataset is attached
OUT = "/kaggle/working/vitok-data"
PRETRAIN_BYTES = 1e10 # UTF-8 bytes of pretraining text (enough for d10 = 1B bpe-nfc tokens incl. packing loss)
TOK_TRAIN_BYTES = 5e8
VOCAB_SIZES = [16000, 32000]  # 32k only to check Gate 1
TRANSITION = 0.9
SUPERBPE_COMMIT = "bbd09768fc28a875cef48e6bdd66e3a17454628e"

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

CODE = Path("/tmp/vitok")
bundles = list(Path("/kaggle/input").rglob("pyproject.toml"))
bundles = [p.parent for p in bundles if (p.parent / "src" / "vitok").exists()]

if bundles:
    shutil.copytree(bundles[0], CODE, dirs_exist_ok=True)
else:
    assert VITOK_REPO, "attach the vitok-code dataset or set VITOK_REPO"
    sh(f"git clone --depth 1 {VITOK_REPO} {CODE}")

sh(f"pip install -q -e {CODE}")
sys.path.insert(0, str(CODE / "src"))  # the running kernel does not see the editable install

In [ ]:
# Step 2: FineWeb-2 vie_Latn -> tok_train.txt, pretraining shards, val shard, test.jsonl, syllables.json
sh(f"python -m vitok.data --out {OUT} --pretrain-bytes {PRETRAIN_BYTES} --tok-train-bytes {TOK_TRAIN_BYTES} --cache-dir /tmp/hf_raw")
shutil.move(f"{OUT}/tok_train.txt", "/tmp/tok_train.txt")  # keep the output dataset small
sh(f"python -m vitok.minimal_pairs --test {OUT}/test.jsonl --syllables {OUT}/syllables.json --out {OUT}/minimal_pairs.jsonl")

In [ ]:
# SuperBPE's forked `tokenizers` (needs Rust), only used by the check in step 3a.
# This replaces the stock tokenizers in THIS notebook only.
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
os.environ["PATH"] = f"{Path.home()}/.cargo/bin:" + os.environ["PATH"]

if not Path("/tmp/superbpe").exists():  # re-running the cell in the same session
    sh("git clone https://github.com/PythonNut/superbpe /tmp/superbpe")
    sh(f"cd /tmp/superbpe && git checkout {SUPERBPE_COMMIT} && git submodule update --init --depth 1")

sh("pip install -q /tmp/superbpe/tokenizers_superbpe/bindings/python")

In [ ]:
# Step 3a: check vitok.superbpe (fast stage 2) against the SuperBPE fork on a 3MB corpus.
# The fork cannot finish stage 2 on the full 500MB, so the full run below uses `fast`.
import vitok.superbpe  # stale-code guard: vitok-code must include the fast stage 2
from vitok.tokenizer_spec import STAGE2_REGEX

assert r"\p{M}" in STAGE2_REGEX, "stale vitok-code: upload the version with the NFD stage-2 regex fix"

with open("/tmp/tok_train.txt", "rb") as fin, open("/tmp/tok_small.txt", "wb") as fout:
    n = 0
    for line in fin:
        fout.write(line)
        n += len(line)
        if n >= 3e6:
            break

for s in ("fork", "fast"):
    shutil.rmtree(f"/tmp/check_work_{s}", ignore_errors=True)
    sh(f"python -m vitok.train_tokenizers --corpus /tmp/tok_small.txt --out /tmp/check-{s} "
       f"--vocab-size 4000 --transition {TRANSITION} --stage2 {s} --workdir /tmp/check_work_{s} > /tmp/check_{s}.log")
print("inherited merges the fork skipped:", open("/tmp/check_fork.log").read().count("not found in queue"))
for cond in ("super-nfc", "super-nfd"):
    fork, fast = (json.load(open(f"/tmp/check-{s}/{cond}/tokenizer.json"))["model"] for s in ("fork", "fast"))
    first_diff = next((i for i, (x, y) in enumerate(zip(fork["merges"], fast["merges"])) if x != y), None)
    print(f"{cond}: fork {len(fork['merges'])} merges, fast {len(fast['merges'])}, first difference at {first_diff}")
    assert fork["merges"] == fast["merges"] and fork["vocab"] == fast["vocab"], "fast stage 2 differs from the fork"
print(open("/tmp/check_fast.log").read())

In [ ]:
# Step 3b: train BPE / SuperBPE x NFC / NFD (stage 2 = vitok.superbpe), then measure compression (Gate 1)
for v in VOCAB_SIZES:
    k = f"{v // 1000}k"
    sh(f"python -m vitok.train_tokenizers --corpus /tmp/tok_train.txt --out {OUT}/tokenizers-{k} "
       f"--vocab-size {v} --transition {TRANSITION} --workdir /tmp/tok_work_{k}")
    sh(f"python -m vitok.compression --tokenizers {OUT}/tokenizers-{k} "
       f"--docs {OUT}/shards/shard_99999.parquet --out {OUT}/compression-{k}.json")

In [ ]:
# Step 3 checks: round-trip 1,000 val docs for every tokenizer; inspect superwords
import pyarrow.parquet as pq
from vitok.hf_tokenizer import HFTokenizer
from vitok.tokenizer_spec import CONDITIONS

docs = pq.read_table(f"{OUT}/shards/shard_99999.parquet").column("text").to_pylist()[:1000]
for v in VOCAB_SIZES:
    k = f"{v // 1000}k"
    for cond in CONDITIONS:
        tok = HFTokenizer.from_directory(f"{OUT}/tokenizers-{k}/{cond}")
        bad = sum(tok.decode(ids) != d for ids, d in zip(tok.encode(docs), docs))
        print(f"{k} {cond:10s} round-trip failures: {bad}/1000")
        assert bad == 0
    comp = json.load(open(f"{OUT}/compression-{k}.json"))
    print(k, "top superwords (super-nfc):", [w for w, _ in comp["super-nfc"]["top_superwords"]])
print("Viet tokens (bpe-nfd 16k):", HFTokenizer.from_directory(f"{OUT}/tokenizers-16k/bpe-nfd").tok.encode("Tiếng Việt").tokens)
print(json.load(open(f"{OUT}/stats.json")))
sh(f"du -sh {OUT}/*")